# 04 Structured Output & Middleware

By the end you can return typed TravelPlan objects, pause sensitive tools with HITL, and cap agent loops.


Set up the project path and reload shared helpers from this repo.


In [1]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


c:\Users\Azooo\langchain-bootcamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\Azooo\langchain-bootcamp


Check which API keys and provider settings are available in `.env`.


In [2]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


ZAI_API_KEY: True
OPENAI_API_KEY: False
ANTHROPIC_API_KEY: False
TAVILY_API_KEY: True
LANGSMITH_API_KEY: True
LLM_PROVIDER: deepseek


In [ ]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


In [4]:
from shared.dataflow import preview, print_agent_dataflow, print_rag_dataflow, print_final_state


Define TravelPlan schema and sensitive book_trip tool. Structured fields and middleware targets start here, print the JSON schema keys you expect the model to fill.


In [5]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
import json

class TravelPlan(BaseModel):
    destination: str
    budget_used: float
    weather_summary: str
    itinerary: list[str]

@tool
def search_destinations(query: str, budget: int) ->  str:
    """Search destinations."""
    return "Lisbon, $700"

@tool
def book_trip(destination: str) ->  str:
    """Book a trip, sensitive."""
    return f"Booked {destination}"

print("TravelPlan schema:")
print(json.dumps(TravelPlan.model_json_schema(), indent=2)[:400], "...")


TravelPlan schema:
{
  "properties": {
    "destination": {
      "title": "Destination",
      "type": "string"
    },
    "budget_used": {
      "title": "Budget Used",
      "type": "number"
    },
    "weather_summary": {
      "title": "Weather Summary",
      "type": "string"
    },
    "itinerary": {
      "items": {
        "type": "string"
      },
      "title": "Itinerary",
      "type": "array"
    }
  } ...


Requires `DEEPSEEK_API_KEY` in `.env` (default provider: DeepSeek).


In [7]:
model = require_llm()
agent = create_agent(
    model=model,
    tools=[search_destinations],
    response_format=TravelPlan,
    system_prompt="Return a structured travel plan."
)
question = "Plan Lisbon under $900"
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print_rag_dataflow(question, "tool/search_destinations: Lisbon, $700")
print_agent_dataflow(result["messages"])
sr = result.get("structured_response")
if sr:
    print("STRUCTURED:", preview(sr.model_dump()))
else:
    print("STRUCTURED: (from last message)", preview(result["messages"][-1].content))


DATAFLOW
1. question: Plan Lisbon under $900
2. retrieve: tool/search_destinations: Lisbon, $700
DATAFLOW
[0] HumanMessage: Plan Lisbon under $900
[1] AIMessage tool_calls: ["search_destinations({'query': 'Lisbon', 'budget': 900})"]
[2] ToolMessage(search_destinations): Lisbon, $700
[3] AIMessage tool_calls: ["TravelPlan({'destination': 'Lisbon', 'budget_used': 700, 'weather_summary': 'Lisbon enjoys a Mediterranean climate with mild, rainy winters and warm, dry summers. Spring and fall are pleasant...)"]
[4] ToolMessage(TravelPlan): Returning structured response: destination='Lisbon' budget_used=700.0 weather_summary='Lisbon enjoys a Mediterranean climate with mild, rainy winters and warm, dry summers. Spring ...
FINAL: Returning structured response: destination='Lisbon' budget_used=700.0 weather_summary='Lisbon enjoys a Mediterranean climate with mild, rainy winters and warm, dry summers. Spring ...
STRUCTURED: {'destination': 'Lisbon', 'budget_used': 700.0, 'weather_summary': 'Lisbon

## Human-in-the-loop (HITL) middleware

**What:** `HumanInTheLoopMiddleware` pauses the agent **before** a sensitive tool runs (here, `book_trip`).

**How (LangChain docs pattern):**
1. `create_agent(..., checkpointer=InMemorySaver(), middleware=[HumanInTheLoopMiddleware(...)])`
2. `invoke` with a `thread_id` until the graph hits an interrupt
3. Read `get_state(...).interrupts[0].value` (tool name + args)
4. Human approves, edits, or rejects in the notebook prompt
5. `invoke(Command(resume={"decisions": [...]}), config=...)` to continue

The cell below runs steps 1 to 5. You will be prompted **in real time** before `book_trip` executes.


In [ ]:
from langgraph.types import Command
from shared.notebook_display import ask_hitl_decision

HITL_CONFIG = {
    "configurable": {"thread_id": "nb04-booking-1"},
    "recursion_limit": 15,
}

hitl_agent = create_agent(
    model=require_llm(),
    tools=[book_trip],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_trip": {
                    "allowed_decisions": ["approve", "reject"],
                }
            }
        )
    ],
    system_prompt="Use book_trip when the user asks to book.",
)

question = "Book a trip to Lisbon for $700"
print("STEP 1: invoke until middleware pauses on book_trip")
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=HITL_CONFIG,
)

snap = hitl_agent.get_state(HITL_CONFIG)
if snap.interrupts:
    print("\nCHECKPOINT (paused before tool runs)")
    print(f"  next nodes: {snap.next}")
    print("\nSTEP 2: your decision at the prompt below")
    resume_payload = ask_hitl_decision(snap.interrupts[0].value)
    result = hitl_agent.invoke(
        Command(resume=resume_payload),
        config=HITL_CONFIG,
    )
else:
    print("No interrupt: model did not call book_trip.")

print_agent_dataflow(result["messages"])


STEP 1: invoke until middleware pauses on book_trip

CHECKPOINT (paused before tool runs)
  next nodes: ('HumanInTheLoopMiddleware.after_model',)

STEP 2: your decision at the prompt below
HUMAN REVIEW (HITL middleware)
  tool: book_trip
  args: {'destination': 'Lisbon'}
  note: Tool execution requires approval
DATAFLOW
[0] HumanMessage: Book a trip to Lisbon for $700
[1] AIMessage tool_calls: ["book_trip({'destination': 'Lisbon'})"]
[2] ToolMessage(book_trip): User rejected the tool call for `book_trip` with id call_00_5O7MWsAp6aSiurm1kCDq8832. The tool was not executed. Do not retry this tool call unless the user explicitly requests it.
[3] AIMessage: It looks like the booking for Lisbon was not able to be completed at this time. Unfortunately, I wasn't provided with details on why it was rejected.  Would you like to try booking...
FINAL: It looks like the booking for Lisbon was not able to be completed at this time. Unfortunately, I wasn't provided with details on why it was rejecte

## Invoke config guardrails

Always pass `thread_id` (required for HITL checkpoints) and `recursion_limit` (caps tool loops). The HITL cell above already uses both via `HITL_CONFIG`.


In [9]:
print("Pass to invoke/stream (same shape as HITL_CONFIG above):")
print(HITL_CONFIG)
print("Without recursion_limit, a confused model can loop tools indefinitely.")
print("Without thread_id, HITL cannot resume the paused checkpoint.")


Pass to invoke/stream:
{'recursion_limit': 15, 'configurable': {'thread_id': 'demo-1'}}
Without this, a confused model can loop tools indefinitely.
